# Notebook 03 — RAG Generation Demo

This notebook covers Part 3 of the assessment:

- Full RAG pipeline: query → retrieve → generate cited answer
- System prompt design for medical context
- Demo on required queries (English + Turkish)
- Bonus: query expansion and re-ranking

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env')

# Check API key
api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    print('⚠ GOOGLE_API_KEY not set. Get a free key at: https://aistudio.google.com/app/apikey')
    print('Then add it to your .env file: GOOGLE_API_KEY=your_key_here')
else:
    print(f'API key configured (ends with: ...{api_key[-4:]})')

# Load corpus
with open('../data/pubmed_articles.json', encoding='utf-8') as f:
    articles = json.load(f)
print(f'Loaded {len(articles)} articles')

In [ ]:
# Build retrievers (uses cached embeddings)
from src.retrieval import build_retrievers

print('Building retrieval indices...')
retrievers = build_retrievers(articles, use_cache=True)
hybrid = retrievers['hybrid']
print('Done.')

## System Prompt Design

Key design decisions for our medical RAG system:

1. **Context-only answers**: Prevents hallucination of clinical recommendations
2. **PMID citations**: Allows doctors to verify claims in original sources
3. **Language mirroring**: Turkish doctors get Turkish answers from English papers
4. **Conservative safety**: Explicitly instructs not to extrapolate beyond text

In [ ]:
from src.rag import SYSTEM_PROMPT
print('System Prompt:')
print('='*60)
print(SYSTEM_PROMPT)
print('='*60)

## Demo Query 1 — English

In [ ]:
from src.rag import generate_answer

query1 = 'What are the latest guidelines for managing type 2 diabetes?'

# Retrieve top-5 with Hybrid RRF
docs1 = hybrid.search(query1, top_k=5)

# Generate answer
result1 = generate_answer(query1, docs1, provider='google', verbose=True)

## Demo Query 2 — Turkish

In [ ]:
query2 = 'Çocuklarda akut otitis media tedavisi nasıl yapılır?'

docs2 = hybrid.search(query2, top_k=5)
result2 = generate_answer(query2, docs2, provider='google', verbose=True)

## Demo Query 3 — Bonus Query

In [ ]:
query3 = 'Iron supplementation dosing for anemia during pregnancy'

docs3 = hybrid.search(query3, top_k=5)
result3 = generate_answer(query3, docs3, provider='google', verbose=True)

---
## Bonus — Query Expansion

Improvement: Use automatic translation to expand Turkish queries before retrieval.
This improves BM25 recall for Turkish queries by generating English equivalents.

In [ ]:
from google import genai as _genai
from google.genai import types as _gtypes

def expand_query(query: str) -> str:
    """
    Bonus improvement: translate/expand Turkish medical queries to English.
    Improves BM25 recall which relies on lexical overlap with English abstracts.
    """
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        return query
    client = _genai.Client(api_key=api_key)
    prompt = f"""If the following medical query is in Turkish, translate it to English. 
If it is already in English, return it unchanged. 
Respond with ONLY the translated/original query, nothing else.

Query: {query}"""
    try:
        resp = client.models.generate_content(
            model="google-default-model",
            contents=prompt,
            config=_gtypes.GenerateContentConfig(max_output_tokens=100, temperature=0.0)
        )
        return (resp.text or query).strip()
    except Exception:
        return query


class QueryExpandingHybridRetriever:
    """
    Bonus: Hybrid retriever with translation-based query expansion for Turkish queries.
    Translates Turkish to English, then fuses BM25(TR) + BM25(EN) + Semantic(TR) via RRF.
    """
    def __init__(self, articles, bm25_r, semantic_r, k=60):
        self.articles = articles
        self.pmid_to_article = {a["pmid"]: a for a in articles}
        self.bm25 = bm25_r
        self.semantic = semantic_r
        self.k = k
        from src.retrieval import reciprocal_rank_fusion
        self.rrf = reciprocal_rank_fusion

    def search(self, query: str, top_k: int = 5) -> list:
        import time
        expanded = expand_query(query)
        if expanded != query:
            print(f"  [QueryExpansion] {query!r}")
            print(f"             --> {expanded!r}")
            time.sleep(0.5)
        ranked_lists = [
            self.bm25.get_ranked_list(query, top_k=50),
            self.semantic.get_ranked_list(query, top_k=50),
        ]
        if expanded != query:
            ranked_lists.append(self.bm25.get_ranked_list(expanded, top_k=50))
        fused = self.rrf(ranked_lists, k=self.k)
        results = []
        for rank, (pmid, score) in enumerate(fused[:top_k]):
            if pmid in self.pmid_to_article:
                art = self.pmid_to_article[pmid].copy()
                art["score"] = score
                art["rank"] = rank + 1
                art["method"] = "hybrid_rrf_expanded"
                results.append(art)
        return results


expanded_hybrid = QueryExpandingHybridRetriever(
    articles, retrievers["bm25"], retrievers["semantic"]
)
print("Query-expanding hybrid retriever built")


In [ ]:
# Compare standard hybrid vs query-expanding hybrid on Turkish query
turkish_query = 'Çölyak hastalığı tanı kriterleri nelerdir?'
print(f'Query: {turkish_query!r}')
print('(English: What are the diagnostic criteria for celiac disease?)\n')

print('Standard Hybrid RRF top-5:')
for r in hybrid.search(turkish_query, top_k=5):
    print(f'  [{r["rank"]}] (score={r["score"]:.5f}) {r["title"][:70]}...')

print('\nQuery-Expanding Hybrid top-5:')
for r in expanded_hybrid.search(turkish_query, top_k=5):
    print(f'  [{r["rank"]}] (score={r["score"]:.5f}) {r["title"][:70]}...')

In [ ]:
# RAG with the expanded hybrid retriever
query4 = 'Çölyak hastalığı tanı kriterleri nelerdir?'
docs4 = expanded_hybrid.search(query4, top_k=5)
result4 = generate_answer(query4, docs4, provider='google', verbose=True)

## Summary

In [ ]:
print('RAG Demo Summary')
print('='*60)
for i, result in enumerate([result1, result2, result3, result4], 1):
    print(f'\nQuery {i}: {result["query"][:60]}...')
    print(f'  Sources retrieved: {len(result["sources"])}')
    print(f'  PMIDs cited in answer: {result.get("cited_pmids", [])}')
    print(f'  Answer length: {len(result["answer"])} chars')